<a href="https://colab.research.google.com/github/pablonvsx/pisi3-ufrpe/blob/main/data-science/notebooks/ML/aquasense_variavel_temporal_lgbm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Variável Temporal — Períodos de 20 Anos

O dataset do AquaSense possui registros entre **1940 e 2023**, cobrindo uma amplitude total de 83 anos.

Nesta etapa, o dataset foi analisado visando a criação de uma variável temporal baseada em **intervalos de 20 anos**, gerando 5 períodos históricos distintos:

| Período | Anos |
|---|---|
| 1940–1959 | Pós-guerra, início da industrialização moderna |
| 1960–1979 | Expansão industrial, primeiras legislações ambientais |
| 1980–1999 | Consolidação de políticas ambientais globais |
| 2000–2019 | Era dos ODM/ODS, monitoramento ambiental intensificado |
| 2020–2023 | Período recente, pós-pandemia |

A escolha de 20 anos foi feita porque:

- Intervalos menores (5 ou 10 anos) gerariam períodos com poucos registros nas décadas de 1940–1970, criando classes desbalanceadas temporalmente.
- Intervalos maiores perderiam a capacidade de capturar mudanças históricas relevantes na qualidade da água.
- Períodos de 20 anos são amplamente utilizados em análises ambientais de longo prazo e coincidem com ciclos de políticas internacionais (como os Objetivos de Desenvolvimento do Milênio e os ODS da ONU).

> **Nota:** O período 2020–2023 é parcial (apenas 4 anos), mas foi mantido por representar um contexto ambiental e social distinto dos demais.

# Preparação do Ambiente

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import lightgbm as lgb

warnings.filterwarnings("ignore")

SEED = 42

print(f"LightGBM version: {lgb.__version__}")

LightGBM version: 4.6.0


In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    amostra_path = "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/amostra_rotulada.parquet"
else:
    amostra_path = "../../dataset/processed/amostra_rotulada.parquet"

df = pd.read_parquet(amostra_path)

print("Amostra carregada com sucesso.")
print(f"Shape: {df.shape}")

Mounted at /content/drive
Amostra carregada com sucesso.
Shape: (141399, 22)


# Variável Temporal

In [ ]:
# PARSE DA DATA E EXTRAÇÃO DO ANO
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
df["ano"] = df["Date"].dt.year

print(f"Ano mínimo: {df['ano'].min()}")
print(f"Ano máximo: {df['ano'].max()}")
print(f"Amplitude total: {df['ano'].max() - df['ano'].min()} anos")
print(f"Datas nulas após parse: {df['ano'].isna().sum()}")

Ano mínimo: 1940
Ano máximo: 2023
Amplitude total: 83 anos
Datas nulas após parse: 0


In [ ]:
# CRIAÇÃO DOS PERÍODOS DE 20 ANOS
bins = [1940, 1960, 1980, 2000, 2020, 2024]
labels = [
    "1940-1959",
    "1960-1979",
    "1980-1999",
    "2000-2019",
    "2020-2023"
]

df["periodo_20"] = pd.cut(
    df["ano"],
    bins=bins,
    labels=labels,
    right=False
)

print("Distribuição por período:")
print(df["periodo_20"].value_counts().sort_index())

Distribuição por período:
periodo_20
1940-1959      2003
1960-1979      5992
1980-1999      4112
2000-2019    116701
2020-2023     12591
Name: count, dtype: int64


# Análise Exploratória da Variável Temporal

In [ ]:
# DISTRIBUIÇÃO DE REGISTROS POR PERÍODO
contagem_periodo = (
    df["periodo_20"]
    .value_counts()
    .sort_index()
    .reset_index()
)
contagem_periodo.columns = ["Período", "Registros"]

fig = px.bar(
    contagem_periodo,
    x="Período",
    y="Registros",
    title="Distribuição de Registros por Período Histórico (20 anos)",
    color="Período",
    text="Registros"
)
fig.update_traces(textposition="outside")
fig.show()

In [ ]:
# DISTRIBUIÇÃO DO CONAMA_STATUS POR PERÍODO
cross = (
    df.groupby(["periodo_20", "conama_status"], observed=True)
    .size()
    .reset_index(name="count")
)

cross["total_periodo"] = cross.groupby("periodo_20")["count"].transform("sum")
cross["proporcao"] = (cross["count"] / cross["total_periodo"] * 100).round(2)

fig = px.bar(
    cross,
    x="periodo_20",
    y="proporcao",
    color="conama_status",
    barmode="stack",
    title="Distribuição do conama_status por Período Histórico (%)",
    labels={
        "periodo_20": "Período",
        "proporcao": "Proporção (%)",
        "conama_status": "Status CONAMA"
    },
    color_discrete_map={
        "Excelente": "#2ecc71",
        "Boa": "#3498db",
        "Atenção": "#f39c12",
        "Crítica": "#e74c3c"
    }
)
fig.show()

In [ ]:
# EVOLUÇÃO DO OXIGÊNIO DISSOLVIDO POR PERÍODO
od_periodo = (
    df.groupby("periodo_20", observed=True)["Dissolved Oxygen (mg/l)"]
    .median()
    .reset_index()
)
od_periodo.columns = ["Período", "OD Mediano (mg/l)"]

fig = px.line(
    od_periodo,
    x="Período",
    y="OD Mediano (mg/l)",
    markers=True,
    title="Evolução Mediana do Oxigênio Dissolvido por Período"
)
fig.add_hline(
    y=5,
    line_dash="dash",
    annotation_text="Mínimo Classe 2 CONAMA"
)
fig.show()

In [ ]:
# EVOLUÇÃO DA DBO POR PERÍODO
dbo_periodo = (
    df.groupby("periodo_20", observed=True)["Biochemical Oxygen Demand (mg/l)"]
    .median()
    .reset_index()
)
dbo_periodo.columns = ["Período", "DBO Mediana (mg/l)"]

fig = px.line(
    dbo_periodo,
    x="Período",
    y="DBO Mediana (mg/l)",
    markers=True,
    title="Evolução Mediana da DBO por Período"
)
fig.add_hline(
    y=5,
    line_dash="dash",
    annotation_text="Máximo Classe 2 CONAMA"
)
fig.show()